# 06 · A/B sample size and duration — Keelpay copilot

Companion of `06_ab_protocol.md` (issue #63). It computes, **before any data exists**, how many tickets and how many weeks the pilot A/B needs, and how sensitive that is to the two numbers we do not know yet (spread of handling time, clustering by agent-week).

Every input is either a cell of `02_sizing.xlsx` (cited `Sheet!Cell`) or a **named assumption** to be replaced by the copilot-off baseline week (§3 of the protocol). No observed Keelpay number exists: the scenario is fictional.

Design reminder: staggered crossover. Agents are split into two groups (stratified by desk and seniority); group 1 is copilot-on in odd weeks, group 2 in even weeks. Conversations are split 50/50 at arrival into a copilot stream and a manual stream; an agent in a copilot-on week only takes copilot-stream tickets. The cluster is the agent-week.

In [1]:
from math import ceil, sqrt
from scipy.stats import norm

# --- From 02_sizing.xlsx (Base column) ---
TICKETS_PER_MONTH = 36_900          # Inputs!D6
AGENTS = 46                         # Inputs!D7
MEAN_MIN = 10.0                     # Inputs!G7, ~10 min hands-on per ticket
TICKETS_PER_AGENT_DAY = 38.2        # Inputs!D11 (volume / agents / 21 working days)
BREAK_EVEN = 0.0593                 # Decision!D56, time saving for zero net value
BEAT_DO_NOTHING = 0.1279            # Decision!D58
BEAT_NEXT_BEST = 0.1332             # Decision!D57
SIZED_SAVING = 0.2026               # Flow!D15, assumed saving at full adoption
SERIOUS_ERR_BASE = 0.000745         # Decision!D59, weighted incremental serious errors

# --- Named assumptions (replace with the baseline week) ---
CV = 1.0      # SD / mean of hands-on minutes per ticket (right-skewed service times)
ICC = 0.02    # share of variance between agent-weeks, after agent and week effects
ALPHA, POWER = 0.05, 0.80
ALLOC = 0.5   # 50/50 streams and 50/50 agents

tickets_per_week = TICKETS_PER_MONTH * 12 / 52
tickets_per_arm_week = tickets_per_week * ALLOC
m = TICKETS_PER_AGENT_DAY * 5       # tickets per agent-week = cluster size
print(f"tickets per week: {tickets_per_week:,.0f} | per arm: {tickets_per_arm_week:,.0f} | per agent-week (m): {m:.0f}")

tickets per week: 8,515 | per arm: 4,258 | per agent-week (m): 191


## 1 · Primary metric: hands-on minutes per ticket

**MDE = 6 % relative (0.6 min on 10 min)**: the smallest effect that changes the decision is the break-even point (5.93 %, `Decision!D56`). Below it the copilot loses money whatever else happens; we want the test to see an effect that size. Two-sided test, α = 0.05, power 0.80.

Individual-level sample size for a difference of means, inflated by the design effect `1 + (m − 1) × ICC` for agent-week clusters.

In [2]:
def n_per_arm(mde_rel, cv=CV, icc=ICC, alpha=ALPHA, power=POWER, m=m):
    z = norm.ppf(1 - alpha / 2) + norm.ppf(power)
    n0 = 2 * z**2 * cv**2 / mde_rel**2          # tickets per arm, independent tickets
    deff = 1 + (m - 1) * icc
    return n0, deff, n0 * deff

MDE = 0.06
n0, deff, n = n_per_arm(MDE)
weeks = n / tickets_per_arm_week
print(f"independent tickets per arm: {n0:,.0f}")
print(f"design effect (m={m:.0f}, ICC={ICC}): {deff:.2f}")
print(f"tickets per arm: {n:,.0f}  ->  {weeks:.1f} weeks of crossover")
print(f"agent-weeks per arm in 6 weeks: {AGENTS * ALLOC * 6:.0f}")

independent tickets per arm: 4,360
design effect (m=191, ICC=0.02): 4.80
tickets per arm: 20,930  ->  4.9 weeks of crossover
agent-weeks per arm in 6 weeks: 138


### Sensitivity: weeks needed, by spread (CV) and clustering (ICC)

The duration is set from this table, not from the single cell above: the pre-registered horizon must hold for plausible values.

In [3]:
cvs = [0.8, 1.0, 1.2, 1.5]
iccs = [0.0, 0.01, 0.02, 0.05]
print("weeks needed for MDE = 6 %  (rows: CV, columns: ICC)")
print("CV \\ ICC " + "".join(f"{i:>8}" for i in iccs))
for cv in cvs:
    row = [n_per_arm(MDE, cv=cv, icc=i)[2] / tickets_per_arm_week for i in iccs]
    print(f"{cv:>9} " + "".join(f"{w:>8.1f}" for w in row))

print()
print("MDE reachable in 6 weeks (CV = 1.0)")
n_6w = tickets_per_arm_week * 6
for icc in iccs:
    deff_i = 1 + (m - 1) * icc
    z = norm.ppf(1 - ALPHA / 2) + norm.ppf(POWER)
    mde = sqrt(2 * z**2 * CV**2 * deff_i / n_6w)
    print(f"  ICC {icc:<5} -> MDE {mde:.1%}")

weeks needed for MDE = 6 %  (rows: CV, columns: ICC)
CV \ ICC      0.0    0.01    0.02    0.05
      0.8      0.7     1.9     3.1     6.9
      1.0      1.0     3.0     4.9    10.8
      1.2      1.5     4.3     7.1    15.5
      1.5      2.3     6.7    11.1    24.2

MDE reachable in 6 weeks (CV = 1.0)
  ICC 0.0   -> MDE 2.5%
  ICC 0.01  -> MDE 4.2%
  ICC 0.02  -> MDE 5.4%
  ICC 0.05  -> MDE 8.0%


**Reading.** With the base assumptions (CV 1.0, ICC 0.02) about 5 weeks suffice; **6 weeks of crossover** (3 copilot-on and 3 copilot-off weeks per agent) is the pre-registered horizon. At ICC 0.05 the 6-week MDE grows to about 8 %, still below the 12.8–13.3 % thresholds that decide the comparison with do-nothing and the next-best project. The horizon is recomputed **once**, from the copilot-off baseline week only (nuisance parameters, no treatment data), and capped at 10 weeks.

## 2 · Guardrails: what the same 6 weeks can and cannot show

In [4]:
n_arm = tickets_per_arm_week * 6
z1 = norm.ppf(1 - ALPHA) + norm.ppf(POWER)   # one-sided, non-inferiority

# QA score of sent replies: non-inferiority, margin 0.25 SD of the QA rubric (assumption)
qa_margin_sd = 0.25
n_qa = 2 * z1**2 / qa_margin_sd**2
print(f"QA score, margin 0.25 SD: {n_qa:.0f} scored replies per arm -> {ceil(n_qa / 6)} per arm per week")

# Repeat contact within 7 days: non-inferiority, baseline 10 % (assumption), margin +1.5 points
p0, margin = 0.10, 0.015
deff_rc = 1 + (m - 1) * 0.01
n_rc = z1**2 * 2 * p0 * (1 - p0) / margin**2 * deff_rc
print(f"repeat contact, margin +1.5 pts: {n_rc:,.0f} tickets per arm (available in 6 weeks: {n_arm:,.0f})")

# Uncaught serious errors: can the test see a doubling of the sizing base?
p1, p2 = SERIOUS_ERR_BASE, 2 * SERIOUS_ERR_BASE
z2 = norm.ppf(1 - ALPHA / 2) + norm.ppf(POWER)
n_se = z2**2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p2 - p1) ** 2
print(f"serious errors, doubling {p1:.3%} -> {p2:.3%}: {n_se:,.0f} tickets per arm "
      f"= {n_se / tickets_per_arm_week:.1f} weeks, and only if every error is found")
print(f"expected serious errors per arm in 6 weeks at the base rate: {n_arm * p1:.0f}")

QA score, margin 0.25 SD: 198 scored replies per arm -> 33 per arm per week
repeat contact, margin +1.5 pts: 14,344 tickets per arm (available in 6 weeks: 25,546)
serious errors, doubling 0.074% -> 0.149%: 31,567 tickets per arm = 7.4 weeks, and only if every error is found
expected serious errors per arm in 6 weeks at the base rate: 19


**Reading.**

- QA score: 198 scored replies per arm are needed (33 a week); we take 35 per arm and per week (210 in total).
- Repeat contact: the 6 weeks are enough for a +1.5 point margin.
- **Uncaught serious errors: not powered.** Seeing a doubling would need about 8 weeks *and* finding every error, which a QA sample does not do. Serious errors are controlled by the stopping rules of the protocol, not by this test (`governance.md` Session 2: a sample of this size cannot certify a rare rate).

## 3 · Sample-ratio check (run every week)

The stream split must stay 50/50. A χ² test with p < 0.001 pauses the analysis until the assignment bug is found.

In [5]:
from scipy.stats import chisquare

def srm_check(n_copilot, n_manual, threshold=0.001):
    stat, p = chisquare([n_copilot, n_manual])
    return p, ("PAUSE: sample-ratio mismatch" if p < threshold else "ok")

# Example: one week at the expected volume, then a 52/48 drift
print(srm_check(4_258, 4_257))
print(srm_check(4_428, 4_087))

(np.float64(0.9913535221189603), 'ok')
(np.float64(0.00021953647815426788), 'PAUSE: sample-ratio mismatch')


## 4 · Summary for the protocol

| Item | Value |
|---|---|
| MDE, primary | 6 % relative (0.6 min), ≈ break-even `Decision!D56` |
| α, power | 0.05 two-sided, 0.80 |
| Tickets per arm | ≈ 21,000 at CV 1.0 / ICC 0.02 |
| Horizon | 1 copilot-off baseline week + 6 crossover weeks (cap 10 after the one re-estimation) |
| QA sample | 35 replies per arm per week |
| Serious errors | not powered; stopping rules instead |

Assumptions to replace with the baseline week: CV of hands-on time, ICC of agent-weeks, repeat-contact base rate, SD of the QA score.